# VoiceRail on a GPU pod

Train voice models and convert songs on a rented GPU (RunPod, Vast.ai, Lambda, Paperspace, any machine with Jupyter and an NVIDIA GPU).

1. Upload this notebook to the pod's Jupyter (or open it from the cloned repository).
2. Run the cells top to bottom. The first run installs Python, PyTorch and the base models (a few GB); later runs start in seconds.
3. Open the address printed by the *Start the web UI* cell (the pod's port proxy for port 8080, or an SSH tunnel).

Everything that is slow to recreate (packages, models, training sessions, outputs) is stored on the persistent volume (`/workspace` when it exists), so stopping and restarting the pod keeps your work.

In [ ]:
# ---- Settings -------------------------------------------------------------
REPO_URL = "https://github.com/MigiSenpai416/VoiceRail.git"  # or your fork
BRANCH = "main"
# ---------------------------------------------------------------------------
import os
import pathlib
import subprocess

VOLUME = pathlib.Path("/workspace" if os.path.isdir("/workspace") else pathlib.Path.home())
REPO = VOLUME / "VoiceRail"
if pathlib.Path("pod.sh").is_file() and pathlib.Path("pyproject.toml").is_file():
    REPO = pathlib.Path.cwd()  # the notebook already lives inside a clone
elif (REPO / "pod.sh").is_file():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print("VoiceRail repository:", REPO)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv || echo "No NVIDIA GPU found: VoiceRail will run on the CPU (slow)."

In [ ]:
# Install uv, Python 3.12, the right PyTorch build for this GPU, and fetch the
# base models (pitch estimator, voice encoder, pretrained RVC v2 at 40 kHz).
!bash pod.sh download

In [ ]:
# Start the web UI in the background and print where to open it.
import re
import subprocess
import time

LOG = "/tmp/voicerail-web.log"

if "webui" in globals() and webui.poll() is None:
    webui.terminate()
    webui.wait()
webui = subprocess.Popen(["bash", "pod.sh"], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
deadline = time.time() + 900
url = None
while time.time() < deadline and webui.poll() is None:
    match = re.search(r"VoiceRail is running at (\S+)", open(LOG, errors="replace").read())
    if match:
        url = match.group(1)
        break
    time.sleep(2)
if webui.poll() is not None:
    print(open(LOG, errors="replace").read()[-4000:])
    raise SystemExit("The web UI stopped; see the log above.")
pod_id = os.environ.get("RUNPOD_POD_ID")
if pod_id:
    print(f"Open: https://{pod_id}-8080.proxy.runpod.net  (expose HTTP port 8080 in the pod template)")
else:
    print("The UI listens on port 8080. Open it through the pod's port mapping, or from your computer:")
    print("  ssh -L 8080:localhost:8080 <pod>   then   http://localhost:8080")
print("Inside the pod:", url or "starting… run the next cell in a minute")

In [ ]:
# Show the latest log lines of the web UI (links, training progress, errors).
print(open(LOG, errors="replace").read()[-6000:])

## Command line (optional)

The same work can be scripted without the UI. Put recordings on the volume (for example `/workspace/recordings/`) and run:

In [ ]:
# !bash pod.sh train MyVoice --data /workspace/recordings --preset songs
# !bash pod.sh convert /workspace/song.mp3 -m MyVoice -p 12
# !bash pod.sh import https://huggingface.co/user/repo/resolve/main/model.zip
!bash pod.sh models

## Your files

- Voice models: `<volume>/voicerail-data/models/<name>/` (`.pth` + `.index`, compatible with RVC WebUI and Applio)
- Converted songs: `<volume>/voicerail-data/outputs/`
- Training sessions and logs: `<volume>/voicerail-data/training/<name>/`

Download them from the Jupyter file browser, or zip a model first:

In [ ]:
import shutil

MODEL = "MyVoice"
data = VOLUME / "voicerail-data"
archive = shutil.make_archive(str(data / "outputs" / MODEL), "zip", data / "models" / MODEL)
print("Model archive:", archive)

In [ ]:
# Stop the web UI (training that is already running continues in its own process).
if "webui" in globals() and webui.poll() is None:
    webui.terminate()
    webui.wait()
    print("Web UI stopped.")